## Build a Simple LLM Application with LCEL

In this quickstart, we build a simple LLM application using LangChain.  
The application translates text from English into another language.

Although this is a minimal example — just a single LLM call combined with prompting — it demonstrates the core building blocks of most LangChain applications.

Many real-world systems start with this exact pattern:
Prompt → LLM → Output

---

## What You Will Learn

After this section, you will understand:

- How to use language models within LangChain
- How to construct prompts using `PromptTemplate`
- How to structure outputs using `OutputParser`
- How to chain components together using LangChain Expression Language (LCEL)
- How to debug and trace your application with LangSmith
- How to deploy your application using LangServe

---

## Why This Matters

Even simple LLM pipelines can power:

- Translators  
- Summarizers  
- Structured data extractors  
- Q&A tools  
- Chat interfaces  

Mastering this foundational pattern is essential before moving to more advanced concepts like RAG, agents, or tool calling.


In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

groq_api_key = os.getenv("GROQ_API_KEY")

In [18]:
from langchain_groq import ChatGroq
model=ChatGroq(model= "llama-3.1-8b-instant", api_key=os.environ.get("GROQ_API_KEY"))
model

ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 8192, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x13da36650>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x13de98e90>, model_name='llama-3.1-8b-instant', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [56]:
from langchain_core.messages import HumanMessage, SystemMessage
messages = [
    SystemMessage(content="Translate the following text into English."),
    HumanMessage(content="Bonjour, comment ça va?")]

result = model.invoke(messages)

In [55]:
from langchain_core.output_parsers import StrOutputParser
output_parser = StrOutputParser()
output_parser.invoke(result)

'The translation of the text is:\n\n"Alors, comment ça va?" \n\nHowever, the original phrase "Casse de toi" is a more casual way to say "Alors, comment ça va?" which can be translated to "So, how are you?" \n\nHere\'s a breakdown:\n\n- "Casse" is an informal way to say "Alors" (so), but it can be translated to "then" or "so" in some contexts.'

In [38]:
# Chaining the components

chain= model | output_parser
print(chain.invoke(messages))

The translation of the text is:

"Hello, how are you?"


In [59]:
# Prompt templates
from langchain_core.prompts import ChatPromptTemplate
generic_template = "Translate the following text into {language}."
prompt = ChatPromptTemplate.from_messages([
    ("system", generic_template),
    ("human", "{text}") # This is the variable for the French text
])  
result1 = prompt.invoke({
    "language": "English", 
    "text": "Bonjour, comment ça va?"
})
print(result1.to_messages())

[SystemMessage(content='Translate the following text into English.', additional_kwargs={}, response_metadata={}), HumanMessage(content='Bonjour, comment ça va?', additional_kwargs={}, response_metadata={})]


In [60]:
result1.to_messages()

[SystemMessage(content='Translate the following text into English.', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='Bonjour, comment ça va?', additional_kwargs={}, response_metadata={})]

In [65]:
chain = prompt | model | output_parser
print(chain.invoke({
    "language": "English", 
    "text": "je suis fatigué"
}))

The translation is:

"I am tired."
